# Исследование игровой индустрии

## Описание проекта

Проект посвящён анализу данных о видеоиграх: платформах, жанрах, годах выпуска, продажах в разных регионах, оценках пользователей и критиков, а также возрастных рейтингах ESRB.

**Цель проекта** — подготовить данные за 2000–2013 годы и выделить основные характеристики игровой индустрии за этот период: популярные платформы, качество данных, структуру оценок и категории игр.

## Данные

В датасете представлены следующие признаки:

- `Name` — название игры;
- `Platform` — игровая платформа;
- `Year of Release` — год выпуска;
- `Genre` — жанр;
- `NA sales`, `EU sales`, `JP sales`, `Other sales` — продажи по регионам, млн копий;
- `Critic Score` — оценка критиков;
- `User Score` — оценка пользователей;
- `Rating` — возрастной рейтинг ESRB.

## План работы

1. Загрузить данные и изучить их структуру.
2. Проверить типы данных, пропуски и дубликаты.
3. Выполнить предобработку: переименовать столбцы, привести типы, обработать пропуски и дубликаты.
4. Отобрать данные за 2000–2013 годы.
5. Разделить игры на категории по оценкам пользователей и критиков.
6. Сформулировать итоговые выводы.

## Используемые инструменты

- Python
- pandas

## Загрузка данных и знакомство с ними

In [6]:
import pandas as pd #выгружаем датасет
df = pd.read_csv('/kaggle/input/datasets/anzhelikanichkova/gamess/new_games.csv')

In [7]:
df.info() #выводим первые строки
df.head(15) #выводим инф-ю о датасете

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.0,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
6,New Super Mario Bros.,DS,2006.0,Platform,11.28,9.14,6.5,2.88,89.0,8.5,E
7,Wii Play,Wii,2006.0,Misc,13.96,9.18,2.93,2.84,58.0,6.6,E
8,New Super Mario Bros. Wii,Wii,2009.0,Platform,14.44,6.94,4.7,2.24,87.0,8.4,E
9,Duck Hunt,NES,1984.0,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN


Полученные данные состоят из 11 столбцов, 16956 строк и соответствуют описанию. В 6 столбцах встречаются пропуски. Типы данных используются верно, кроме столбцей "Year of Release" (данные в нем нужно привести к типу с целыми числами (int64)), "EU sales", "JP sales", "User Score", "Rating" - в них тип данных нужно изменить на вещественный тип (float64).
Столбец "Rating" для лаконичности переименуем в "Rating ESRB"

## Проверка ошибок в данных и их предобработка

In [8]:
df.columns #выводим названия столбцов

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [9]:
df.columns = df.columns.str.lower().str.replace(' ', '_') #меняем стиль написания на snake case
df = df.rename(columns={'rating': 'rating_esrb'}) #меняем непонятное названние столбца
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating_esrb'],
      dtype='object')


### Типы данных

Предположительные причины некорретных типов данных:

1. человеческие ошибки
2. ошибки при автоматическом определении типа

In [10]:
column_convert = ['eu_sales', 'jp_sales', 'user_score'] # переменная для изменений типа 
for column in column_convert: #цикл преобразует строковые значения в NaN
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['year_of_release'] = df['year_of_release'].astype('Int32') #явно приводим столбец к Int для сохранения NaN

In [11]:
null_sum = df.isna().sum() #считаем абсолютное кол-во пропусков в каждом столбце
display(null_sum)

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating_esrb        6871
dtype: int64

In [12]:
relative_value = null_sum / len(df)*100 #считаем относительное кол-во пропусков в каждом столбце
display(relative_value)

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating_esrb        40.522529
dtype: float64

Пропуски харктерны для столбцов, в которых хранятся данные о рейтинге игр: 
1. critic_score (возможно, потому что не на все игры есть рецензии критиков)
2. user_score (возможно, потому что не каждый пользователь оценивает игры)
3. rating_esrb (возможно, потому что оцениваемые игры вышли раньше, чем появился рейтинг организации ESRB)

Незначительный (в сравнении предыдущих столбцов) процент пропусков приходится на стоблец с годами выпуска игр. Это возможно, потому что произошла ошибка (техническая либо человеческий фактор). Очень малое кол-во пропусков в столбцах с названнием игры (0.01%), в столбцах с продажами в Японии (0.02%) и Европе (0.04%)

Процент пропусков в столбце с годами выпуска игр невысокий (1.6%), поэтому строки с пропусками мы удалим, чтобы не искажать данные. С этой же целью относительно выбросов, в столбцах с рейтингом заполним пропуски медианным значением. Строки с пропущенными названиями мы тоже удалим, это не повлияет на анализ. Пропуски в столбах с продажами заполним средним значением от года выхода игры.

In [13]:
df = df.dropna(subset=['year_of_release', 'name']) #удаляем пропуски в столбце с годами и названиями

In [14]:
df[['critic_score', 'user_score']] = df[['critic_score', 'user_score']].fillna(df[['critic_score', 'user_score']].median()) #заполняем пропуски в столбцах с рейтингом

In [15]:
for sales_column in ['eu_sales', 'jp_sales']: #цикл перебирает нужные столбцы, в которых есть пропуски, заполняет медианным значнеием в зависимости от года
    avg_by_year = df.groupby('year_of_release')[sales_column].median()
    for year in avg_by_year.index:
        df.loc[(df['year_of_release'] == year) & (df[sales_column].isna()), sales_column] = avg_by_year[year]

In [16]:
df['rating_esrb'] = df['rating_esrb'].fillna(df['rating_esrb'].mode()[0]) #заполняем пропуски самым часто встречающимся рейтингом

### Явные и неявные дубликаты в данных

Изучаем уникальные значения в категориальных данных

In [17]:
categorical_columns = ['genre', 'platform', 'rating_esrb', 'year_of_release'] #создаем переменную с нужными столбцами
for column in categorical_columns: #цикл выводит уникальные значения в каждом столбце
    print(f"{column}")
    print(df[column].unique())

genre
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
platform
['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
rating_esrb
['E' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
year_of_release
<IntegerArray>
[2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004,
 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994,
 1982, 2016, 2003, 1986, 2000, 1995, 1991, 1981, 1987, 1980, 1983]
Length: 37, dtype: Int32


In [18]:
df['genre'] = df['genre'].str.lower() #неявные дубликаты обнаружены в столбце с жанрами. проведем нормализацию данных
                                       #к нижнему регистру

In [19]:
duplicates_all = df[df.duplicated(keep=False)] #в этой переменной сохраняем все явные дублирующиеся строки
print(f"Всего дублирующихся строк: {len(duplicates_all)}") #находим кол-во дубликатов
display(duplicates_all)

Всего дублирующихся строк: 470


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating_esrb
267,Batman: Arkham Asylum,PS3,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
268,Batman: Arkham Asylum,PS3,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
367,James Bond 007: Agent Under Fire,PS2,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
368,James Bond 007: Agent Under Fire,PS2,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
716,God of War: Ascension,PS3,2013,action,1.23,0.63,0.04,0.35,80.0,7.5,M
...,...,...,...,...,...,...,...,...,...,...,...
16799,Transformers: Prime,Wii,2012,action,0.00,0.01,0.00,0.00,71.0,7.5,E
16911,Metal Gear Solid V: The Definitive Experience,XOne,2016,action,0.01,0.00,0.00,0.00,71.0,7.5,M
16912,Metal Gear Solid V: The Definitive Experience,XOne,2016,action,0.01,0.00,0.00,0.00,71.0,7.5,M
16939,The Longest 5 Minutes,PSV,2016,action,0.00,0.00,0.01,0.00,71.0,7.5,E


Кол-во найденных явных дубликатов - 470 строк. Явные дубликаты по своей природе являются абсолютно идентичными, поэтому применим методы pandas для их устранения

In [20]:
df = df.drop_duplicates(keep='first') #удаляем явные дубликаты с сохранением первой строки

В исходном датасете было 16 956 строк. Воспользуемся этим числом

In [21]:
before_len = 16956
after_len = 16956 - len(df) #вычисляем абсолютное значение удаленных строк путем вычитания нанешнего кол-ва строк из изначального
relative_len = round (after_len / before_len * 100, 2) #вычисляем относительное кол-во удаленных строк, округляем
display(f"Всего строк удалено: {after_len}")
display(f"Кол-во удаленных строк относительно изначального (%): {relative_len}")

'Всего строк удалено: 512'

'Кол-во удаленных строк относительно изначального (%): 3.02'

На этапе предобработки данных мы:
1. Нормализовали стиль написания столбцов
2. Преобразовали данные в нужный тип
3. Обработали пропуски и дубликаты в данных

Большинство пропусков удалось заполнить. Количество дубликатов оказалось небольшим(470 строк), как и количество удаленных строк (512 или 3.02% от всех строк). Этот процент говорит о том, что данные почти не потеряны и результат не будет искажен.


## Фильтрация данных

In [22]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)] #в новом датафрейме сохраняем данные за 2000-2013 года

## Категоризация данных

Разделим игры на категории по оценкам пользователей и критиков. Это поможет быстрее сравнивать игры не только по числовым оценкам, но и по качественным группам.

Категории по оценкам пользователей:

- низкая оценка: от 0 до 3;
- средняя оценка: от 3 до 8;
- высокая оценка: от 8 до 10.

Категории по оценкам критиков:

- низкая оценка: от 0 до 30;
- средняя оценка: от 30 до 80;
- высокая оценка: от 80 до 100.

In [23]:
df_actual = df_actual.copy() #pandas дал предупреждение SettingWithCopyWarning, создала копию

In [24]:
df_actual['user_score_category'] = pd.cut( #категоризируем игры по оценкам пользователей
    df_actual['user_score'],
    bins=[0, 3, 8, 10], #расставляем границы
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], #именуем категории
    right=False)  # правый край интервала не включается (кроме последнего)

In [25]:
df_actual['critic_score_category'] = pd.cut( #все аналогично коду выше, только по оценке критиков
    df_actual['critic_score'],
    bins=[0, 30, 80, 100],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False)

In [26]:
display("По оценкам пользователей") #выводим текст
display(df_actual.groupby('user_score_category', observed=False)['user_score'].count()) #группируем по категориям и считаем количество игр в них

'По оценкам пользователей'

user_score_category
низкая оценка       116
средняя оценка    10379
высокая оценка     2286
Name: user_score, dtype: int64

In [27]:
display("По оценкам критиков") #аналогично коду выше, только для оценки критиков
display(df_actual.groupby('critic_score_category', observed=False)['critic_score'].count())

'По оценкам критиков'

critic_score_category
низкая оценка        55
средняя оценка    11034
высокая оценка     1692
Name: critic_score, dtype: int64

In [28]:
count_game = df_actual['platform'].value_counts() #считаем кол-во игр по платформам. 
#данный метод сортирует по умолчанию от большего к меньшему

In [29]:
display(count_game.head(7)) #поэтому просто выводим первые 7 платформ

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

---

## Итоговый вывод

В ходе выполнения проекта мы провели полную предобработку данных об игровых платформах, подготовили срез за период 2000–2013 годы и добавили новые категориальные признаки на основе оценок.

Описание этапов:
1. Загрузили данные, исправили типы столбцов (продажи, оценки, год выпуска), привели названия к удобному формату
2. Обработали пропуски: удалили строки без года и названия, остальные заполнили медианами и модой
3. Убрали дубликаты — явные (470 шт.), обработали неявные в жанрах, что позволило не исказить данные при анализе
4. Отфильтровали период 2000–2013 годов в отдельный датафрейм `df_actual`, в него же вошли новые столбцы (`user_score_category` и `critic_score_category`) с категориями оценки критиков и пользователей (низкая, средняя, высокая)

За данный временной период самыми популярными по количеству выходов игр платформами оказались:
- PS2 (2127 игр)
- DS (2120)
- Wii (1275)
Подготовленный датасет можно использовать для дальнейшего анализа рынка: сравнения платформ, жанров, региональных продаж, пользовательских и критических оценок.